In [0]:
%run ../00-common/01.environment_config

In [0]:
results_table    = f"{catalog_name}.{silver_schema}.results"
sprint_table = f"{catalog_name}.{silver_schema}.sprints"

target_table = f"{catalog_name}.{gold_schema}.results_fact"

In [0]:
from pyspark.sql.functions import *

In [0]:
result_df = spark.read.table(results_table)
result_df = (
    result_df.withColumn('session_type', lit('RACE'))
             .drop('race_date','race_name','ingestion_timestamp','source_files')
)


In [0]:
sprints_df = spark.read.table(sprint_table)
sprints_df = (
    sprints_df.withColumn('session_type', lit('SPRINT'))
             .drop('race_date','race_name','ingestion_timestamp','source_files')
) 

In [0]:
sprint_result_df=(
    sprints_df
    .unionByName(result_df)
)

In [0]:
sprint_result_fact_df=(
    sprint_result_df
    .withColumn('is_win',col("finsh_position")==1)
    .withColumn('is_podium',col("finsh_position").between(1,3))
    .withColumn('has_points', col("points")>0)
)

In [0]:
(
    sprint_result_fact_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(target_table)
)